# Varlen Transformer v2 (SM75 Turing / Tesla T4) — Pure THD, Paged KV Cache & Fused Eager GQA

End-to-end guide on **NVIDIA Tesla T4 (SM75)** (Google Colab / Kaggle / local CUDA GPU):
1. Verify GPU, CUDA, PyTorch environment and initialize **NVIDIA CUTLASS**.
2. Build the CUDA/CuTe extension (`mha_fwd`, `mha_bwd`, `kv_cache`, `mha_decode`, `gqa_fused`) and package a reusable `.whl` in `wheels/`.
3. Verify numerical correctness (**Pure THD Fwd/Bwd & Strided Slices**, **Paged KV Cache & Split-K Decode**, **Fused Eager Self/Cross-Attention**, and **`FakeTensor` schemas**).
4. Run **Training Benchmark** (Pure Eager Fused Varlen vs `torch.compile` Batched SDPA Baseline) and save `assets/training_benchmark.png`.
5. Run **Inference Benchmark** (Paged Decode `Lq=1` & E2E Prefill/Generation) and save `assets/inference_benchmark.png`.
6. Run **NVIDIA Nsight Compute (`ncu`)** to export `.ncu-rep` reports (`fwd`, `bwd`, `kv`, `decode`) into `benchmark/`.

## 1. Check GPU & Prepare Source / CUTLASS

In [ ]:
import os, sys, subprocess

if not os.path.isfile("setup.py") and not os.path.isdir("Varlen-Attention"):
    !git clone --recursive --depth 1 https://github.com/CBG6682/Varlen-Attention.git

if os.path.isdir("Varlen-Attention"):
    %cd Varlen-Attention
elif os.path.basename(os.getcwd()).lower() == "benchmark":
    %cd ..

print("Working directory:", os.getcwd())

import torch
print("PyTorch:", torch.__version__, "| CUDA available:", torch.cuda.is_available(), "| CUDA runtime:", torch.version.cuda)
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0), "| Capability:", torch.cuda.get_device_capability(0))

!nvidia-smi

In [ ]:
%%bash
set -e
if [ ! -f "third_party/cutlass/include/cute/tensor.hpp" ]; then
    echo "Initializing NVIDIA CUTLASS submodule in third_party/cutlass..."
    git submodule update --init --depth 1 third_party/cutlass
else
    echo "CUTLASS headers already exist in third_party/cutlass/include."
fi

## 2. Build Extension & Package Wheel (`wheels/`)
Checks for a compatible **v2** `.whl` in `wheels/` (verifying all `THD` / `Paged KV` / `Fused GQA` symbols in `_C`); otherwise builds cleanly from source and packages the `.whl`.

In [ ]:
%%bash
set -euo pipefail
export MAX_JOBS=2
export VERBOSE=1
export TORCH_EXTENSIONS_VERBOSE=1

pip install -q ninja wheel setuptools

check_v2_ext() {
    python - << "EOF"
import varlen_transformer as vt
assert vt.HAS_CUDA_EXT
for sym in (
    "fused_mha_varlen_fwd",
    "fused_mha_varlen_bwd",
    "prefill_to_paged_kv",
    "paged_kv_append",
    "fused_mha_varlen_decode",
    "fused_gqa_varlen_fwd",
    "fused_gqa_varlen_prefill",
    "fused_gqa_varlen_bwd",
    "fused_cross_gqa_varlen_fwd",
    "fused_cross_gqa_varlen_bwd",
    "fused_gqa_varlen_decode",
):
    assert hasattr(vt._C, sym), f"Missing {sym}"
EOF
}

if ls wheels/varlen_transformer-*.whl 1>/dev/null 2>&1; then
    echo "[INFO] Found wheel in wheels/, verifying v2 compatibility..."
    FORCE_REBUILD=0 python setup.py build_ext --inplace
    FORCE_REBUILD=0 python -m pip install -q --no-deps --force-reinstall wheels/varlen_transformer-*.whl
    if check_v2_ext 2>/dev/null; then
        echo "[OK] Prebuilt v2 wheel is active and matches this Python/PyTorch environment."
        ls -lh wheels/*.whl varlen_transformer/_C*.so
        exit 0
    fi
    echo "[WARN] Existing wheel is stale or ABI-incompatible. Rebuilding v2 from source..."
    rm -f wheels/varlen_transformer-*.whl varlen_transformer/_C*.so
fi

export FORCE_REBUILD=1
# Build extension (C++20 with C++17 fallback on legacy PyTorch) + verbose nvcc/cxx logs (-v)
python setup.py build_ext --inplace -v
# Package wheel (.whl) with prebuilt _C*.so into wheels/
FORCE_REBUILD=0 python -m pip wheel -v --no-deps --no-build-isolation -w wheels .
# Install package in editable mode pointing to current source
FORCE_REBUILD=0 python -m pip install -v --no-deps --no-build-isolation -e .
check_v2_ext
ls -lh wheels/*.whl varlen_transformer/_C*.so

## 3. Correctness Verification (v2 Pure THD + Paged KV + Fused Self/Cross GQA)
Verifies:
- Raw `fused_mha_varlen_fwd` / `fused_mha_varlen_bwd` on Pure `(Total_L, H, 64)` layout & zero-copy strided `qkv` slices
- `prefill_to_paged_kv`, `paged_kv_append` (crossing `PAGE_SIZE = 16` boundaries), and Split-K `fused_mha_varlen_decode` with shuffled `block_table`
- End-to-End `VarlenFlashGQA` Self-Attention, Cross-Attention, Prefill + 20-step Paged Decode, and `FakeTensor` registrations

In [ ]:
!python tests/test_correctness.py

## 4. Training Benchmark (Pure Eager Fused Varlen vs `torch.compile` Batched Baseline)
Compares **Pure Eager `VarlenFlashGQA`** (C++ `at::mm` + Pure THD CUDA kernels, no `torch.compile` needed) directly against **`BatchedFlashGQABaselineSpeed` + `torch.compile`** (TorchInductor) and Eager Batched SDPA.

In [ ]:
!python tests/bench_training.py \
    --batch-sizes 4 8 16 \
    --max-lens 512 1024 2048 \
    --distributions skewed uniform \
    --save-plot assets/training_benchmark.png

In [ ]:
from IPython.display import Image, display
import os
if os.path.exists("assets/training_benchmark.png"):
    display(Image("assets/training_benchmark.png"))

## 5. Inference Benchmark (Paged Decode `Lq=1` & E2E: Compiled Baseline vs Pure Eager Varlen)
Compares **`BatchedFlashGQABaselineSpeed` + `torch.compile`** (`SDPBackend.EFFICIENT_ATTENTION` / xFormers CUTLASS) against **Pure Eager `VarlenFlashGQA` (v2)** using **C++ Fused Linear + Paged KV Cache (`PAGE_SIZE=16`) + Adaptive Direct/Split-K Decoding**.

> **Technical Note on Baseline SDPA on T4 (`SM75`):** PyTorch SDPA with a `bool` `pad_mask` falls back to the slow `MATH` backend. Therefore, the benchmark uses an **additive `float16` mask (`0.0` / `-inf`)** and aligns the last dimension `L` to a **multiple of 8** with explicit strides `(aligned_len, 0, 0, 1)` to activate **`SDPBackend.EFFICIENT_ATTENTION`** (xFormers) for a fair baseline.

In [ ]:
# Pass --e2e-quick to run a quick 5-config E2E subset
!python tests/bench_inference.py \
    --batch-sizes 1 8 32 64 \
    --max-lens 128 256 1024 2048 4096 \
    --num-heads-q 32 \
    --num-heads-kv 8 \
    --distributions skewed uniform \
    --gen-steps 32 \
    --save-plot assets/inference_benchmark.png

In [ ]:
from IPython.display import Image, display
import os
if os.path.exists("assets/inference_benchmark.png"):
    display(Image("assets/inference_benchmark.png"))

## 6. Nsight Compute Profiling (`.ncu-rep` in `benchmark/`)
Collects hardware performance counters (`SM Throughput`, `DRAM Bandwidth`, `Achieved Occupancy`, `Register/Thread`) with **embedded CUDA/C++ source (`--import-source yes`)** into `benchmark/`:
- `benchmark/mha_fwd.ncu-rep` (Pure THD Forward across `short_b1`, `short_b16`, `skewed_b4`, `long_b1`)
- `benchmark/mha_bwd.ncu-rep` (Pure THD Backward + `kernel_cast_dq_f32_to_f16`)
- `benchmark/kv_cache.ncu-rep` (`kernel_prefill_to_paged_kv` & `kernel_paged_kv_append`)
- `benchmark/mha_decode.ncu-rep` (Paged Split-K & Direct Flash Decoding)

> Note: On some cloud containers without hardware counter permissions for `ncu`, the command may report `ERR_NVGPUCTRPERM`.

In [ ]:
%%bash
set -e
if ! command -v ncu &> /dev/null; then
    echo "[INFO] NVIDIA Nsight Compute (ncu) not found in PATH."
    exit 0
fi

NCU_SCRIPT='import sys, torch, varlen_transformer as vt
mode = sys.argv[1]
dev = torch.device("cuda")
cases = [
    ("short_b1", [128]),
    ("short_b16", [128] * 16),
    ("skewed_b4", [256, 512, 1024, 2048]),
    ("long_b1", [4096]),
]
rope = vt.RotaryPositionalEmbedding(64, max_seq_len=4200, device=dev)
cos, sin = rope.get_cos_sin(device=dev, dtype=torch.float16)
for cname, lens in cases:
    cu = torch.tensor([0] + torch.cumsum(torch.tensor(lens), 0).tolist(), dtype=torch.int32, device=dev)
    total_l = int(cu[-1].item())
    max_l = max(lens)
    B = len(lens)
    k = torch.randn((total_l, 4, 64), device=dev, dtype=torch.float16)
    v = torch.randn((total_l, 4, 64), device=dev, dtype=torch.float16)
    if mode == "fwd":
        q = torch.randn((total_l, 16, 64), device=dev, dtype=torch.float16)
        vt.fused_mha_varlen_fwd(q, k, v, cu, cu, max_l, cos=cos, sin=sin, is_causal=True)
    elif mode == "bwd":
        q = torch.randn((total_l, 16, 64), device=dev, dtype=torch.float16)
        o, lse = vt.fused_mha_varlen_fwd(q, k, v, cu, cu, max_l, cos=cos, sin=sin, is_causal=True)
        do = torch.randn_like(o)
        vt.fused_mha_varlen_bwd(do, q, k, v, o, lse, cu, cu, None, max_l, cos=cos, sin=sin, is_causal=True)
    elif mode == "kv":
        cache = vt.VarlenKVCache.from_packed(k, v, cu, max_seqlen_k=max_l)
        nk = torch.randn((B, 4, 64), device=dev, dtype=torch.float16)
        nv = torch.randn((B, 4, 64), device=dev, dtype=torch.float16)
        vt.paged_kv_append(nk, nv, cache.k_cache, cache.v_cache, cache.block_table, cache.cache_seqlens)
    elif mode == "decode":
        cache = vt.VarlenKVCache.from_packed(k, v, cu, max_seqlen_k=max_l)
        qd = torch.randn((B, 16, 64), device=dev, dtype=torch.float16)
        vt.fused_mha_varlen_decode(qd, cache.k_cache, cache.v_cache, cache.block_table, cache.cache_seqlens, max_l, cos=cos, sin=sin)
        if cname == "skewed_b4":
            vt.fused_mha_varlen_decode(qd, cache.k_cache, cache.v_cache, cache.block_table, cache.cache_seqlens, max_l, cos=cos, sin=sin, use_splitk=False)
            vt.fused_mha_varlen_decode(qd, cache.k_cache, cache.v_cache, cache.block_table, cache.cache_seqlens, max_l, cos=cos, sin=sin, use_splitk=True)
torch.cuda.synchronize()'

echo "$NCU_SCRIPT" > benchmark/_ncu_profile_target.py

ncu --set full --import-source yes --source-folders .,src,include -k "regex:kernel_fused_mha_varlen_fwd" -f -o "benchmark/mha_fwd" python benchmark/_ncu_profile_target.py fwd || true
ncu --set full --import-source yes --source-folders .,src,include -k "regex:(kernel_fused_mha_varlen_bwd|kernel_cast_dq_f32_to_f16)" -f -o "benchmark/mha_bwd" python benchmark/_ncu_profile_target.py bwd || true
ncu --set full --import-source yes --source-folders .,src,include -k "regex:kernel_(prefill_to_paged_kv|paged_kv_append)" -f -o "benchmark/kv_cache" python benchmark/_ncu_profile_target.py kv || true
ncu --set full --import-source yes --source-folders .,src,include -k "regex:kernel_fused_mha_varlen_decode" -f -o "benchmark/mha_decode" python benchmark/_ncu_profile_target.py decode || true

rm -f benchmark/_ncu_profile_target.py
ls -lh benchmark/*.ncu-rep 2>/dev/null || echo "No .ncu-rep files generated."